# Colab Generation Experiment

Final retrieval prediction JSONL을 그대로 사용해서 Hugging Face 모델로 답변을 생성하고, 사람이 채점할 수 있는 CSV를 만드는 노트북입니다.

실행 순서: GPU 확인 -> repo clone/pull -> 패키지 설치 -> Google Drive mount -> Drive 입력 확인/로컬 복사 -> dry-run -> RUN_LIMIT=20 생성 -> 전체 생성 -> Drive 저장.

OpenAI API, Chroma DB, embedding index는 사용하지 않습니다.

In [11]:
# 1. Experiment config
from pathlib import Path

REPO_URL = "https://github.com/beomsookim1020/chatbot.git"
REPO_BRANCH = "colab-generation"
PROJECT_DIR = Path("/content/chatbot")

DRIVE_INPUT_ROOT = Path("/content/drive/MyDrive/chatbot_colab_inputs")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/chatbot_colab_outputs")

PREDICTION_REL = Path(
    "outputs/predictions/"
    "74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_"
    "soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl"
)
CHUNK_SIDECAR_REL = Path("indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.json")

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
RUN_LIMIT = 20
MAX_NEW_TOKENS = 384
CONTEXT_MAX_CHARS = 1000

print("PROJECT_DIR:", PROJECT_DIR)
print("DRIVE_INPUT_ROOT:", DRIVE_INPUT_ROOT)
print("DRIVE_OUTPUT_ROOT:", DRIVE_OUTPUT_ROOT)
print("MODEL_NAME:", MODEL_NAME)
print("RUN_LIMIT:", RUN_LIMIT)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("CONTEXT_MAX_CHARS:", CONTEXT_MAX_CHARS)

PROJECT_DIR: /content/chatbot
DRIVE_INPUT_ROOT: /content/drive/MyDrive/chatbot_colab_inputs
DRIVE_OUTPUT_ROOT: /content/drive/MyDrive/chatbot_colab_outputs
MODEL_NAME: Qwen/Qwen2.5-3B-Instruct
RUN_LIMIT: 20
MAX_NEW_TOKENS: 384
CONTEXT_MAX_CHARS: 1000


In [12]:
# 2. Colab GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("GPU runtime is not active. In Colab, use Runtime > Change runtime type > GPU, then rerun.")

subprocess.run(["nvidia-smi"], check=True)
print("GPU runtime is active.")

GPU runtime is active.


In [13]:
# 3. Clone or pull colab-generation branch
import shutil
import subprocess

def run_cmd(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)

if (PROJECT_DIR / ".git").exists():
    run_cmd(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(["git", "checkout", REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, PROJECT_DIR])

def ensure_jsonl_chunk_sidecar_support(project_dir: Path) -> None:
    target = project_dir / "src/generation/context_enricher.py"
    text = target.read_text(encoding="utf-8")
    if "def _load_chunk_records" in text:
        print("JSON/JSONL chunk sidecar loader already supported.")
        return
    text = text.replace(
        "    data = json.loads(chunk_path.read_text(encoding=\"utf-8\"))\n",
        "    data = _load_chunk_records(chunk_path)\n",
    )
    helper = '''
def _load_chunk_records(path: Path) -> list[dict[str, Any]]:
    text = path.read_text(encoding="utf-8")
    if not text.strip():
        return []
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and isinstance(data.get("chunks"), list):
        return data["chunks"]
    raise ValueError(f"Unsupported chunk sidecar format: {path}")
'''
    text = text.replace("\n\ndef enrich_retrieved_contexts(\n", "\n\n" + helper + "\ndef enrich_retrieved_contexts(\n")
    target.write_text(text, encoding="utf-8")
    print("Patched JSON/JSONL chunk sidecar loader.")

ensure_jsonl_chunk_sidecar_support(PROJECT_DIR)
print("Project ready:", PROJECT_DIR)

$ git fetch origin colab-generation
$ git checkout colab-generation
$ git pull --ff-only origin colab-generation
Patched JSON/JSONL chunk sidecar loader.
Project ready: /content/chatbot


In [14]:
# 4. Install packages
# The openai package may be installed because the repo imports it, but this notebook never calls OpenAI APIs.
import subprocess
import sys

packages = [
    "-r", str(PROJECT_DIR / "requirements.txt"),
    "transformers>=4.45.0",
    "accelerate",
    "sentencepiece",
    "safetensors",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Packages installed. OpenAI API is not used.")

Packages installed. OpenAI API is not used.


In [15]:
# 5. Google Drive mount
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# 6. Check required input files on Google Drive
DRIVE_EVAL_DIR = DRIVE_INPUT_ROOT / "data/eval"
DRIVE_PREDICTIONS = DRIVE_INPUT_ROOT / PREDICTION_REL
REQUESTED_DRIVE_CHUNK_SIDECAR = DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL
REQUESTED_DRIVE_CHUNK_SIDECAR_JSONL = REQUESTED_DRIVE_CHUNK_SIDECAR.with_suffix(".jsonl")

# The eval loader uses the canonical 01-25 batches by default.
required_eval_csvs = [DRIVE_EVAL_DIR / f"eval_batch_{idx:02d}.csv" for idx in range(1, 26)]
missing_eval_csvs = [path for path in required_eval_csvs if not path.exists()]
all_eval_csvs = sorted(DRIVE_EVAL_DIR.glob("*.csv"))

chunk_sidecar_candidates = [REQUESTED_DRIVE_CHUNK_SIDECAR, REQUESTED_DRIVE_CHUNK_SIDECAR_JSONL]
index_root = DRIVE_INPUT_ROOT / "indexes"
if index_root.exists():
    chunk_sidecar_candidates.extend(path for path in sorted(index_root.glob("**/chunks.json")) if path not in chunk_sidecar_candidates)
    chunk_sidecar_candidates.extend(path for path in sorted(index_root.glob("**/chunks.jsonl")) if path not in chunk_sidecar_candidates)

DRIVE_CHUNK_SIDECAR = next((path for path in chunk_sidecar_candidates if path.exists()), None)
USE_CONTEXT_ENRICHMENT = DRIVE_CHUNK_SIDECAR is not None

missing = []
if missing_eval_csvs:
    missing.append("Missing canonical eval CSVs:\n" + "\n".join(str(path) for path in missing_eval_csvs[:10]))
    if len(missing_eval_csvs) > 10:
        missing.append(f"... and {len(missing_eval_csvs) - 10} more eval CSVs")
if not DRIVE_PREDICTIONS.exists():
    missing.append(str(DRIVE_PREDICTIONS))

print("Eval CSV count:", len(all_eval_csvs))
print("Canonical eval CSVs present:", len(required_eval_csvs) - len(missing_eval_csvs), "/", len(required_eval_csvs))
for path in all_eval_csvs[:10]:
    print(" -", path)
print("Predictions exists:", DRIVE_PREDICTIONS.exists(), DRIVE_PREDICTIONS)
print("Requested chunk sidecar exists:", REQUESTED_DRIVE_CHUNK_SIDECAR.exists(), REQUESTED_DRIVE_CHUNK_SIDECAR)
print("Requested jsonl sidecar exists:", REQUESTED_DRIVE_CHUNK_SIDECAR_JSONL.exists(), REQUESTED_DRIVE_CHUNK_SIDECAR_JSONL)
if DRIVE_CHUNK_SIDECAR:
    print("Using chunk sidecar:", DRIVE_CHUNK_SIDECAR)
else:
    print("Chunk sidecar not found. Generation will continue without context enrichment.")
    print("Searched candidates:")
    for path in chunk_sidecar_candidates[:20]:
        print(" -", path)

if missing:
    raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

print("Required Drive inputs are present.")


Eval CSV count: 38
Canonical eval CSVs present: 25 / 25
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_01.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_02.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_03.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_04.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_05.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_06.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_07.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_08.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_09.csv
 - /content/drive/MyDrive/chatbot_colab_inputs/data/eval/eval_batch_10.csv
Predictions exists: True /content/drive/MyDrive/chatbot_colab_inputs/outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hns

In [17]:
# 7. Copy Drive inputs to Colab local project paths
import shutil

LOCAL_EVAL_DIR = PROJECT_DIR / "data/eval"
LOCAL_PREDICTIONS = PROJECT_DIR / PREDICTION_REL
LOCAL_CHUNK_SIDECAR_REL = CHUNK_SIDECAR_REL.with_suffix(DRIVE_CHUNK_SIDECAR.suffix) if USE_CONTEXT_ENRICHMENT else None
LOCAL_CHUNK_SIDECAR = PROJECT_DIR / LOCAL_CHUNK_SIDECAR_REL if LOCAL_CHUNK_SIDECAR_REL else None

if LOCAL_EVAL_DIR.exists():
    shutil.rmtree(LOCAL_EVAL_DIR)
LOCAL_EVAL_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_EVAL_DIR, LOCAL_EVAL_DIR)

LOCAL_PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_PREDICTIONS, LOCAL_PREDICTIONS)

if USE_CONTEXT_ENRICHMENT:
    LOCAL_CHUNK_SIDECAR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CHUNK_SIDECAR, LOCAL_CHUNK_SIDECAR)

print("Local eval dir:", LOCAL_EVAL_DIR)
print("Local predictions:", LOCAL_PREDICTIONS)
if LOCAL_CHUNK_SIDECAR:
    print("Local chunk sidecar:", LOCAL_CHUNK_SIDECAR)
else:
    print("Local chunk sidecar: disabled")


Local eval dir: /content/chatbot/data/eval
Local predictions: /content/chatbot/outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl
Local chunk sidecar: /content/chatbot/indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl


In [18]:
# 8. Generation helper
import csv
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_DIR = PROJECT_DIR / "outputs/generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def rel(path: Path) -> str:
    return path.relative_to(PROJECT_DIR).as_posix()

def result_paths(label: str) -> dict[str, Path]:
    csv_name = "colab_manual_review.csv" if label == "colab_qwen25_3b_full" else f"{label}_manual_review.csv"
    return {
        "jsonl": OUTPUT_DIR / f"{label}.jsonl",
        "md": OUTPUT_DIR / f"{label}_review.md",
        "csv": OUTPUT_DIR / csv_name,
    }

def run_generation(label: str, limit: int, dry_run: bool = False) -> dict[str, Path]:
    paths = result_paths(label)
    cmd = [
        sys.executable,
        "scripts/generate_answer_samples.py",
        "--predictions", PREDICTION_REL.as_posix(),
        "--eval-dir", "data/eval",
        "--canonical-only",
        "--limit", str(limit),
        "--model", MODEL_NAME,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--context-max-chars", str(CONTEXT_MAX_CHARS),
        "--output", rel(paths["jsonl"]),
        "--review-output", rel(paths["md"]),
    ]
    if USE_CONTEXT_ENRICHMENT and LOCAL_CHUNK_SIDECAR:
        cmd.extend(["--chunk-sidecar", LOCAL_CHUNK_SIDECAR_REL.as_posix()])
    else:
        cmd.append("--no-context-enrichment")
    if dry_run:
        cmd.append("--dry-run")

    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)
    write_manual_review_csv(paths["jsonl"], paths["csv"])
    print("Manual review CSV:", paths["csv"])
    return paths

def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

def write_manual_review_csv(jsonl_path: Path, csv_path: Path) -> None:
    rows = read_jsonl(jsonl_path)
    fieldnames = [
        "id",
        "type",
        "difficulty",
        "question_type",
        "question",
        "generated_answer",
        "ground_truth_answer",
        "ground_truth_docs",
        "retrieved_docs",
        "evidence_candidates",
        "field_candidates",
        "guardrail_confidence",
        "guardrail_warnings",
        "latency_ms",
        "generation_model",
        "human_correctness",
        "evidence_grounded",
        "failure_type",
        "review_memo",
    ]
    with csv_path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            guardrails = row.get("guardrails") or {}
            writer.writerow({
                "id": row.get("id"),
                "type": row.get("type"),
                "difficulty": row.get("difficulty"),
                "question_type": row.get("question_type"),
                "question": row.get("question"),
                "generated_answer": row.get("answer"),
                "ground_truth_answer": row.get("ground_truth_answer"),
                "ground_truth_docs": json.dumps(row.get("ground_truth_docs"), ensure_ascii=False),
                "retrieved_docs": json.dumps(row.get("retrieved_docs") or [], ensure_ascii=False),
                "evidence_candidates": json.dumps((row.get("evidence_sentences") or [])[:5], ensure_ascii=False),
                "field_candidates": json.dumps(row.get("field_candidates") or {}, ensure_ascii=False),
                "guardrail_confidence": guardrails.get("confidence"),
                "guardrail_warnings": json.dumps(guardrails.get("warnings") or [], ensure_ascii=False),
                "latency_ms": row.get("latency_ms"),
                "generation_model": row.get("generation_model"),
                "human_correctness": "",
                "evidence_grounded": "",
                "failure_type": "",
                "review_memo": "",
            })


In [19]:
# 9. Dry-run: build prompts and contexts without loading the model
DRY_RUN_FILES = run_generation("colab_dry_run", limit=RUN_LIMIT, dry_run=True)
dry_rows = read_jsonl(DRY_RUN_FILES["jsonl"])
assert dry_rows, "Dry-run produced no rows."

first = dry_rows[0]
print("Dry-run rows:", len(dry_rows))
print("First ID:", first.get("id"))
print("Question:", first.get("question"))
print("Question type:", first.get("question_type"))
print("Prompt preview:\n")
print(first.get("prompt", "")[:4000])

$ /usr/bin/python3 scripts/generate_answer_samples.py --predictions outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl --eval-dir data/eval --canonical-only --limit 20 --model Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --context-max-chars 1000 --output outputs/generation/colab_dry_run.jsonl --review-output outputs/generation/colab_dry_run_review.md --chunk-sidecar indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl --dry-run
Manual review CSV: /content/chatbot/outputs/generation/colab_dry_run_manual_review.csv
Dry-run rows: 20
First ID: Q001
Question: 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
Question type: budget
Prompt preview:

너는 RFP 문서 기반 QA assistant다.

반드시 지켜야 할 규칙:
1. 제공된 Context 안의 정보만 사용한다.
2. Context에 없으면 추측하지 말고 "문서에서 확인할 수 없습니다"라고 답한다.
3. 금액, 날짜, 기간, 공고번호는 원문 표현을 우선 보존한다.
4. 답변에는 반드시 근거 문서명과 근거 문장을 함께 제시한다.
5. 서로 다른 후보가 있으면 하나로 단정하지 말고 후보를 나눠 말한다.
6. 기초

In [20]:
# 10. RUN_LIMIT=20 generation test with Qwen2.5-3B-Instruct
LIMIT20_FILES = run_generation("colab_qwen25_3b_limit20", limit=RUN_LIMIT, dry_run=False)
limit20_rows = read_jsonl(LIMIT20_FILES["jsonl"])
empty_answers = [row.get("id") for row in limit20_rows if not str(row.get("answer") or "").strip()]
assert len(limit20_rows) == RUN_LIMIT, f"Expected {RUN_LIMIT} rows, got {len(limit20_rows)}"
assert not empty_answers, f"Empty answers: {empty_answers[:10]}"

print("Generated rows:", len(limit20_rows))
print("Manual review CSV:", LIMIT20_FILES["csv"])
print("First generated answer:\n")
print(limit20_rows[0].get("answer", "")[:2000])

$ /usr/bin/python3 scripts/generate_answer_samples.py --predictions outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl --eval-dir data/eval --canonical-only --limit 20 --model Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --context-max-chars 1000 --output outputs/generation/colab_qwen25_3b_limit20.jsonl --review-output outputs/generation/colab_qwen25_3b_limit20_review.md --chunk-sidecar indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl
Manual review CSV: /content/chatbot/outputs/generation/colab_qwen25_3b_limit20_manual_review.csv
Generated rows: 20
Manual review CSV: /content/chatbot/outputs/generation/colab_qwen25_3b_limit20_manual_review.csv
First generated answer:

[답변]
한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 문서에서 확인할 수 없습니다.

[근거]
- 문서명: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
- 근거 문장: [문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp | 사업명: [재공고]차세대 통합정보시스템(ERP) 구축 | 발주기관: 한국가스공

In [21]:
# 11. Full generation run. Run this only after reviewing the RUN_LIMIT=20 output.
if "LIMIT20_FILES" not in globals():
    raise RuntimeError("Run the RUN_LIMIT=20 generation cell first.")

FULL_FILES = run_generation("colab_qwen25_3b_full", limit=0, dry_run=False)
full_rows = read_jsonl(FULL_FILES["jsonl"])
assert full_rows, "Full generation produced no rows."

print("Full generated rows:", len(full_rows))
print("Final manual review CSV:", FULL_FILES["csv"])
print("Final JSONL:", FULL_FILES["jsonl"])
print("Final MD review:", FULL_FILES["md"])

$ /usr/bin/python3 scripts/generate_answer_samples.py --predictions outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl --eval-dir data/eval --canonical-only --limit 0 --model Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --context-max-chars 1000 --output outputs/generation/colab_qwen25_3b_full.jsonl --review-output outputs/generation/colab_qwen25_3b_full_review.md --chunk-sidecar indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl
Manual review CSV: /content/chatbot/outputs/generation/colab_manual_review.csv
Full generated rows: 500
Final manual review CSV: /content/chatbot/outputs/generation/colab_manual_review.csv
Final JSONL: /content/chatbot/outputs/generation/colab_qwen25_3b_full.jsonl
Final MD review: /content/chatbot/outputs/generation/colab_qwen25_3b_full_review.md


In [22]:
# 12. Save generated jsonl, md, csv outputs to Google Drive
import shutil

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
copied = []
for path in sorted(OUTPUT_DIR.glob("colab_*")):
    if path.suffix.lower() not in {".jsonl", ".md", ".csv"}:
        continue
    dest = DRIVE_OUTPUT_ROOT / path.name
    shutil.copy2(path, dest)
    copied.append(dest)

print("Copied outputs:")
for path in copied:
    print(" -", path)

assert copied, "No generation outputs were copied."
print("Drive output root:", DRIVE_OUTPUT_ROOT)

Copied outputs:
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_dry_run.jsonl
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_dry_run_manual_review.csv
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_dry_run_review.md
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_manual_review.csv
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_qwen25_3b_full.jsonl
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_qwen25_3b_full_review.md
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_qwen25_3b_limit20.jsonl
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_qwen25_3b_limit20_manual_review.csv
 - /content/drive/MyDrive/chatbot_colab_outputs/colab_qwen25_3b_limit20_review.md
Drive output root: /content/drive/MyDrive/chatbot_colab_outputs
